# Peta Nasional Ketimpangan Akses Air Minum dan Sanitasi

**Proyek Akhir Data Wrangling — Tahun Data 2025**

Notebook ini menggunakan data indikator yang sudah dibersihkan pada notebook utama, yaitu `outputs/clean_data_kabupaten_kota.csv`. Data indikator tidak dibaca ulang dari file mentah.

GeoJSON BIG dibaca untuk memperoleh geometri batas kabupaten/kota seluruh Indonesia. Notebook kemudian membuat dua peta interaktif:

1. Combined Deprivation Index (CDI).
2. Kategori isu dominan: air minum, sanitasi, seimbang, atau data tidak lengkap.

Geometri disederhanakan khusus untuk tampilan peta. Nilai indikator dan analisis pada CSV tidak diubah.

In [5]:
from pathlib import Path

# Cari folder proyek mulai dari lokasi kerja notebook dan folder induknya.
folder_kandidat = [Path.cwd(), *Path.cwd().parents]

PROJECT_DIR = next(
    (
        folder
        for folder in folder_kandidat
        if (folder / "data" / "raw").is_dir()
        and (folder / "outputs").is_dir()
    ),
    None
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Folder proyek tidak ditemukan. Pastikan notebook berada di "
        "folder proyek atau salah satu subfoldernya, dan folder proyek "
        "memiliki data/raw serta outputs."
    )

RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DATA_PATH = OUTPUT_DIR / "clean_data_kabupaten_kota.csv"
GEOJSON_PATH = RAW_DIR / "batas_kabupaten_kota.geojson"

print("Folder proyek:", PROJECT_DIR)
print("CSV data bersih:", DATA_PATH)
print("GeoJSON batas wilayah:", GEOJSON_PATH)

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"CSV data bersih tidak ditemukan:\n{DATA_PATH}\n"
        "Jalankan notebook utama terlebih dahulu agar CSV tersebut dibuat."
    )

if not GEOJSON_PATH.is_file():
    raise FileNotFoundError(
        f"GeoJSON tidak ditemukan:\n{GEOJSON_PATH}"
    )

Folder proyek: d:\Kuliah\Data Wrangling\Project
CSV data bersih: d:\Kuliah\Data Wrangling\Project\outputs\clean_data_kabupaten_kota.csv
GeoJSON batas wilayah: d:\Kuliah\Data Wrangling\Project\data\raw\batas_kabupaten_kota.geojson


In [2]:
import re
import json
import pandas as pd
import geopandas as gpd
import folium
import numpy as np

from branca.colormap import LinearColormap
from IPython.display import display

# Kode dibaca sebagai teks agar formatnya tetap, misalnya 11.01.
df_indikator = pd.read_csv(
    DATA_PATH,
    dtype={"kode_kabupaten": "string"},
    encoding="utf-8-sig"
)

print("Jumlah baris data indikator:", len(df_indikator))
print("Kolom:", df_indikator.columns.tolist())

kolom_wajib = [
    "kode_kabupaten",
    "nama_kabupaten_resmi",
    "nama_provinsi",
    "akses_air",
    "akses_sanitasi",
    "deprivation_air",
    "deprivation_sanitasi",
    "CDI",
    "isu_dominan",
]

kolom_hilang = [
    kolom for kolom in kolom_wajib
    if kolom not in df_indikator.columns
]

if kolom_hilang:
    raise ValueError(
        "Kolom berikut tidak ditemukan pada CSV data bersih: "
        + ", ".join(kolom_hilang)
    )

# Rapikan format kode dan pastikan indikator terbaca sebagai angka.
df_indikator["kode_kabupaten"] = (
    df_indikator["kode_kabupaten"]
    .astype("string")
    .str.strip()
)

kolom_numerik = [
    "akses_air",
    "akses_sanitasi",
    "deprivation_air",
    "deprivation_sanitasi",
    "CDI",
]

for kolom in kolom_numerik:
    df_indikator[kolom] = pd.to_numeric(
        df_indikator[kolom],
        errors="coerce"
    )

print("Jumlah kode kosong:",
      df_indikator["kode_kabupaten"].isna().sum())
print("Jumlah kode kabupaten unik:",
      df_indikator["kode_kabupaten"].nunique())
print("Jumlah CDI tersedia:",
      df_indikator["CDI"].notna().sum())

Jumlah baris data indikator: 514
Kolom: ['nama_wilayah', 'kunci_nama_final', 'akses_air', 'akses_sanitasi', '_merge', 'kode_kabupaten', 'nama_kabupaten_resmi', 'kode_provinsi', 'nama_provinsi', 'status_referensi', 'deprivation_air', 'deprivation_sanitasi', 'CDI', 'isu_dominan', 'cluster']
Jumlah kode kosong: 0
Jumlah kode kabupaten unik: 514
Jumlah CDI tersedia: 512


In [6]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import json

from branca.colormap import LinearColormap
from IPython.display import display

kolom_peta = [
    "kode_kabupaten",
    "nama_kabupaten_resmi",
    "nama_provinsi",
    "akses_air",
    "akses_sanitasi",
    "deprivation_air",
    "deprivation_sanitasi",
    "CDI",
    "isu_dominan",
]

df_indikator = pd.read_csv(
    DATA_PATH,
    usecols=kolom_peta,
    dtype={"kode_kabupaten": "string"},
    encoding="utf-8-sig"
)

# Rapikan kode wilayah.
df_indikator["kode_kabupaten"] = (
    df_indikator["kode_kabupaten"]
    .astype("string")
    .str.strip()
)

# Pastikan indikator numerik terbaca sebagai angka.
kolom_numerik = [
    "akses_air",
    "akses_sanitasi",
    "deprivation_air",
    "deprivation_sanitasi",
    "CDI",
]

for kolom in kolom_numerik:
    df_indikator[kolom] = pd.to_numeric(
        df_indikator[kolom],
        errors="coerce"
    )

print("Jumlah baris data bersih:", len(df_indikator))
print("Jumlah kode unik:", df_indikator["kode_kabupaten"].nunique())
print("CDI tersedia:", df_indikator["CDI"].notna().sum())
print("CDI kosong:", df_indikator["CDI"].isna().sum())

display(df_indikator.head())

Jumlah baris data bersih: 514
Jumlah kode unik: 514
CDI tersedia: 512
CDI kosong: 2


,akses_air,akses_sanitasi,kode_kabupaten,nama_kabupaten_resmi,nama_provinsi,deprivation_air,deprivation_sanitasi,CDI,isu_dominan
0,97.28,92.55,11.05,Kabupaten Aceh Barat,Aceh,2.72,7.45,5.085,Sanitasi
1,93.60,71.50,11.12,Kabupaten Aceh Barat Daya,Aceh,6.40,28.50,17.450,Sanitasi
2,93.98,90.81,11.06,Kabupaten Aceh Besar,Aceh,6.02,9.19,7.605,Sanitasi
3,92.40,83.17,11.14,Kabupaten Aceh Jaya,Aceh,7.60,16.83,12.215,Sanitasi
4,90.47,78.42,11.01,Kabupaten Aceh Selatan,Aceh,9.53,21.58,15.555,Sanitasi


In [7]:
import re

# Fungsi perbaikan geometri dari Shapely.
try:
    from shapely import make_valid
except ImportError:
    from shapely.validation import make_valid

geojson_awal = gpd.read_file(GEOJSON_PATH)

print("Jumlah fitur awal:", len(geojson_awal))
print("CRS GeoJSON:", geojson_awal.crs)

if geojson_awal.crs is None:
    raise ValueError(
        "CRS GeoJSON tidak tercatat. Periksa metadata file GeoJSON."
    )

if "KDPKAB" not in geojson_awal.columns:
    raise ValueError(
        "Kolom KDPKAB tidak ditemukan pada GeoJSON."
    )

geojson_awal["kode_kabupaten"] = (
    geojson_awal["KDPKAB"]
    .astype("string")
    .str.strip()
)

# Kode kabupaten/kota berbentuk dua digit, titik, dua digit.
pola_kode = r"\d{2}\.\d{2}"

mask_kode_valid = geojson_awal["kode_kabupaten"].str.fullmatch(
    pola_kode,
    na=False
)

geojson_tanpa_kode_valid = geojson_awal.loc[
    ~mask_kode_valid
].copy()

geojson_kode_valid = geojson_awal.loc[
    mask_kode_valid
].copy()

print("Fitur dengan kode valid:", len(geojson_kode_valid))
print("Fitur tanpa kode valid:", len(geojson_tanpa_kode_valid))

kolom_geometri = ["kode_kabupaten", "geometry"]

# Simpan nama dari GeoJSON jika kolomnya tersedia.
for kolom in ["WADMKK", "WADMPR"]:
    if kolom in geojson_kode_valid.columns:
        kolom_geometri.append(kolom)

geojson_kode_valid = geojson_kode_valid[kolom_geometri].copy()

if geojson_kode_valid.geometry.isna().any():
    jumlah_kosong = geojson_kode_valid.geometry.isna().sum()
    raise ValueError(
        f"Ada {jumlah_kosong} geometri kosong pada GeoJSON."
    )

# Perbaiki geometri yang tidak valid.
jumlah_invalid_awal = (
    ~geojson_kode_valid.geometry.is_valid
).sum()

print("Geometri tidak valid sebelum perbaikan:", jumlah_invalid_awal)

if jumlah_invalid_awal > 0:
    geojson_kode_valid["geometry"] = (
        geojson_kode_valid.geometry.map(make_valid)
    )

jumlah_invalid_akhir = (
    ~geojson_kode_valid.geometry.is_valid
).sum()

print("Geometri tidak valid setelah perbaikan:", jumlah_invalid_akhir)

# Beberapa kabupaten/kota dapat terdiri dari lebih dari satu fitur.
# Gabungkan fitur-fitur itu berdasarkan kode wilayah.
geo_nasional = geojson_kode_valid.dissolve(
    by="kode_kabupaten",
    aggfunc="first"
).reset_index()

# Folium membutuhkan koordinat lintang-bujur.
if geo_nasional.crs.to_string() != "EPSG:4326":
    geo_nasional = geo_nasional.to_crs("EPSG:4326")

print("Jumlah geometri setelah digabung:", len(geo_nasional))
print("Jumlah kode geometri unik:",
      geo_nasional["kode_kabupaten"].nunique())
print("CRS untuk pemetaan:", geo_nasional.crs)

Jumlah fitur awal: 548
CRS GeoJSON: EPSG:4326
Fitur dengan kode valid: 520
Fitur tanpa kode valid: 28
Geometri tidak valid sebelum perbaikan: 4
Geometri tidak valid setelah perbaikan: 0
Jumlah geometri setelah digabung: 514
Jumlah kode geometri unik: 514
CRS untuk pemetaan: EPSG:4326


In [8]:
# Gabungkan berdasarkan kode wilayah, bukan nama.
gdf_nasional = geo_nasional.merge(
    df_indikator,
    on="kode_kabupaten",
    how="left",
    validate="one_to_one",
    indicator="_status_join_peta"
)

print("Hasil pencocokan batas dan data indikator:")
display(
    gdf_nasional["_status_join_peta"]
    .value_counts()
    .to_frame("jumlah")
)

# Geometri yang tidak menemukan baris pada CSV.
geo_tanpa_indikator = gdf_nasional.loc[
    gdf_nasional["_status_join_peta"] == "left_only",
    ["kode_kabupaten"]
].copy()

# Kode pada CSV yang tidak memiliki geometri.
kode_geo = set(geo_nasional["kode_kabupaten"].dropna())
kode_data = set(df_indikator["kode_kabupaten"].dropna())
kode_tanpa_geometri = sorted(kode_data - kode_geo)

print("Geometri tanpa data indikator:", len(geo_tanpa_indikator))
print("Kode data tanpa geometri:", len(kode_tanpa_geometri))

if not geo_tanpa_indikator.empty:
    display(geo_tanpa_indikator.head(20))

if kode_tanpa_geometri:
    print("Contoh kode tanpa geometri:", kode_tanpa_geometri[:20])

if gdf_nasional["CDI"].notna().sum() == 0:
    raise ValueError(
        "Tidak ada nilai CDI yang berhasil digabungkan. "
        "Periksa format kode pada CSV dan GeoJSON."
    )

Hasil pencocokan batas dan data indikator:


,jumlah
_status_join_peta,
both,514
left_only,0
right_only,0


Geometri tanpa data indikator: 0
Kode data tanpa geometri: 0


In [9]:
# Proyeksi bermeter untuk penyederhanaan geometri.
gdf_sederhana = gdf_nasional.to_crs("EPSG:6933").copy()

gdf_sederhana["geometry"] = (
    gdf_sederhana.geometry.simplify(
        tolerance=800,
        preserve_topology=True
    )
)

# Kembalikan ke CRS lintang-bujur untuk Folium.
gdf_sederhana = gdf_sederhana.to_crs("EPSG:4326")

print("Jumlah wilayah pada layer peta:", len(gdf_sederhana))
print("Geometri disederhanakan khusus untuk tampilan.")

Jumlah wilayah pada layer peta: 514
Geometri disederhanakan khusus untuk tampilan.


In [10]:
# Buat nilai CDI aman untuk GeoJSON: nilai kosong menjadi null.
gdf_sederhana["CDI_peta"] = (
    pd.to_numeric(gdf_sederhana["CDI"], errors="coerce")
    .astype(object)
    .where(gdf_sederhana["CDI"].notna(), None)
)

# Konversi GeoDataFrame menjadi objek GeoJSON.
geojson_peta = json.loads(
    gdf_sederhana.to_json(drop_id=True)
)

print(
    "Jumlah fitur GeoJSON untuk peta:",
    len(geojson_peta["features"])
)

Jumlah fitur GeoJSON untuk peta: 514


In [15]:
peta_cdi = folium.Map(
    location=[-2.5, 118],
    zoom_start=4,
    tiles=None,
    control_scale=True
)

# CDI secara teoritis berada pada rentang 0–100.
skala_cdi = LinearColormap(
    colors=["#ffffb2", "#fecc5c", "#fd8d3c", "#e31a1c"],
    vmin=0,
    vmax=100,
    caption="Combined Deprivation Index (CDI)"
)

def gaya_cdi(feature):
    nilai = feature["properties"].get("CDI_peta")

    if nilai is None:
        warna = "#bdbdbd"
    else:
        warna = skala_cdi(float(nilai))

    return {
        "fillColor": warna,
        "color": "#555555",
        "weight": 0.35,
        "fillOpacity": 0.78,
    }

folium.GeoJson(
    data=geojson_peta,
    name="CDI kabupaten/kota",
    style_function=gaya_cdi,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "nama_kabupaten_resmi",
            "nama_provinsi",
            "akses_air",
            "akses_sanitasi",
            "CDI",
        ],
        aliases=[
            "Kabupaten/Kota:",
            "Provinsi:",
            "Akses air minum (%):",
            "Akses sanitasi (%):",
            "CDI:",
        ],
        localize=True,
        sticky=False,
    ),
).add_to(peta_cdi)

skala_cdi.add_to(peta_cdi)

minx, miny, maxx, maxy = gdf_sederhana.total_bounds
peta_cdi.fit_bounds([[miny, minx], [maxy, maxx]])

FILE_PETA_CDI = OUTPUT_DIR / "peta_CDI_nasional_2025.html"
peta_cdi.save(FILE_PETA_CDI)

print("Peta CDI nasional tersimpan di:")
print(FILE_PETA_CDI)

Peta CDI nasional tersimpan di:
d:\Kuliah\Data Wrangling\Project\outputs\peta_CDI_nasional_2025.html


In [14]:
warna_isu = {
    "Air minum": "#2b83ba",
    "Sanitasi": "#d7191c",
    "Seimbang": "#fdae61",
    "Data tidak lengkap": "#bdbdbd",
}

def gaya_isu(feature):
    kategori = feature["properties"].get("isu_dominan")

    return {
        "fillColor": warna_isu.get(kategori, "#bdbdbd"),
        "color": "#555555",
        "weight": 0.35,
        "fillOpacity": 0.78,
    }

peta_isu = folium.Map(
    location=[-2.5, 118],
    zoom_start=4,
    tiles=None,
    control_scale=True
)

folium.GeoJson(
    data=geojson_peta,
    name="Kategori isu dominan",
    style_function=gaya_isu,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "nama_kabupaten_resmi",
            "nama_provinsi",
            "deprivation_air",
            "deprivation_sanitasi",
            "CDI",
            "isu_dominan",
        ],
        aliases=[
            "Kabupaten/Kota:",
            "Provinsi:",
            "Deprivation air (%):",
            "Deprivation sanitasi (%):",
            "CDI:",
            "Isu dominan:",
        ],
        localize=True,
        sticky=False,
    ),
).add_to(peta_isu)

legenda_isu = """
<div style="
    position: fixed;
    bottom: 35px;
    left: 35px;
    z-index: 9999;
    background: white;
    padding: 10px 12px;
    border: 1px solid #777;
    border-radius: 4px;
    font-size: 14px;
">
<b>Isu dominan</b><br>
<span style="color:#2b83ba;">■</span> Air minum<br>
<span style="color:#d7191c;">■</span> Sanitasi<br>
<span style="color:#fdae61;">■</span> Seimbang<br>
<span style="color:#bdbdbd;">■</span> Data tidak lengkap
</div>
"""

peta_isu.get_root().html.add_child(
    folium.Element(legenda_isu)
)

minx, miny, maxx, maxy = gdf_sederhana.total_bounds
peta_isu.fit_bounds([[miny, minx], [maxy, maxx]])

FILE_PETA_ISU = OUTPUT_DIR / "peta_isu_dominan_nasional_2025.html"
peta_isu.save(FILE_PETA_ISU)

print("Peta kategori isu dominan nasional tersimpan di:")
print(FILE_PETA_ISU)

Peta kategori isu dominan nasional tersimpan di:
d:\Kuliah\Data Wrangling\Project\outputs\peta_isu_dominan_nasional_2025.html
